# Geo-Nexus v3.2 — P4 Maharashtra Evaluation Source Staging

## Why this notebook exists

The published dataset `sumit07125/geonexus-mh-v3` is the **DAPT train dataset**. Its authoritative build contract contains the DAPT training arrays, metadata, and train-only normalization statistics; it does not contain the Pune/Satara/Vidarbha test imagery required for P4 evaluation.

P4 therefore needs a small, explicit **evaluation-source dataset** containing the exact 30 MH-VAL patches, 30 MH-ADAPT patches, and the current 110-patch verified MH-TEST imagery matched to their verified label manifests.

This notebook builds that source dataset from the existing authoritative processed archive and verified-label release. It does not generate, interpolate, or invent any missing labels.

**Audit revision:** Cell 5 resolves source records by `(zone, split, row, col)` because the authoritative preprocessing metadata stores coordinates rather than requiring a `stem` field.

## Authoritative inputs

The Colab Drive locations below are the locations used by the project's existing Maharashtra verification/release notebook:

```text
MyDrive/geonexus_v3_processed/geonexus_v3_processed.tar
MyDrive/geonexus_v3_processed/verification/verified/
```

The verified release contains:

- `mh_val_labels.npy` + `mh_val_manifest.json` — 30
- `mh_adapt_labels.npy` + `mh_adapt_manifest.json` — 30
- `mh_test_partial_labels.npy` + `mh_test_partial_manifest.json` — 110

The 110 test patches are 40 Pune dry + 40 Satara dry + 30 Vidarbha. The excluded 20 blind + 30 monsoon masks are not synthesized.
### Important manifest detail

The authoritative verified manifests store the patch `stem` but do not necessarily include a `zone` field. The P4 staging code therefore derives the zone from the authoritative stem prefix (`pune_`, `satara_`, or `vidarbha_`) and rejects disagreements when an explicit zone is present.


In [1]:
# ============================================================
# CELL 1 — environment, Drive paths, frozen release contract
# ============================================================

from pathlib import Path
import json, tarfile, re, shutil, hashlib, subprocess, sys, os, time
from collections import Counter, defaultdict

import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/geonexus_v3_processed')
ARCHIVE = DRIVE_ROOT / 'geonexus_v3_processed.tar'
VERIFIED = DRIVE_ROOT / 'verification' / 'verified'
LOCAL_ROOT = Path('/content/geonexus_p4_eval_source_v3_2')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_ID = 'sumit07125/geonexus-mh-p4-eval-v3-2'
UPLOAD_TO_KAGGLE = True

EXPECTED = {
    'mh_val': {'count': 30, 'zones': {'pune': 15, 'satara': 15}},
    'mh_adapt': {'count': 30, 'zones': {'pune': 15, 'satara': 15}},
    'mh_test': {'count': 110, 'zones': {'pune': 40, 'satara': 40, 'vidarbha': 30}},
}
PATCH = 128
CHANNELS = 17
IGNORE = 255
PATCH_SHAPE = (2, CHANNELS, PATCH, PATCH)
EXPECTED_ARRAY_SHAPE = PATCH_SHAPE
EXPECTED_PATCH_SHAPE = PATCH_SHAPE
ZONE_NAMES = ('pune', 'satara', 'vidarbha')
SOURCE_SPLITS = ('train', 'test')
SPLITS = SOURCE_SPLITS

for p, label in [(ARCHIVE, 'processed archive'), (VERIFIED, 'verified release directory')]:
    if not p.exists():
        raise FileNotFoundError(
            f'{label} not found:\n{p}\n\n'
            'This is the authoritative source artifact required to build the P4 evaluation dataset.'
        )

required_verified = [
    'mh_val_labels.npy', 'mh_val_manifest.json',
    'mh_adapt_labels.npy', 'mh_adapt_manifest.json',
    'mh_test_partial_labels.npy', 'mh_test_partial_manifest.json',
]
missing = [name for name in required_verified if not (VERIFIED / name).is_file()]
if missing:
    raise FileNotFoundError('Verified release is incomplete:\n' + '\n'.join(f'  - {x}' for x in missing))

print('=' * 78)
print('Geo-Nexus v3.2 — P4 evaluation source staging')
print('=' * 78)
print('Archive :', ARCHIVE)
print('Verified:', VERIFIED)
print('Output  :', LOCAL_ROOT)
print('Kaggle  :', KAGGLE_DATASET_ID)


Mounted at /content/drive
Geo-Nexus v3.2 — P4 evaluation source staging
Archive : /content/drive/MyDrive/geonexus_v3_processed/geonexus_v3_processed.tar
Verified: /content/drive/MyDrive/geonexus_v3_processed/verification/verified
Output  : /content/geonexus_p4_eval_source_v3_2
Kaggle  : sumit07125/geonexus-mh-p4-eval-v3-2


In [2]:
# ============================================================
# CELL 2 — verify the current 170-patch verified label release
# ============================================================

def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

STEM_RE = re.compile(r'^(pune|satara|vidarbha)_(\d+)_(\d+)$', re.IGNORECASE)

def parse_stem(stem: str):
    text = str(stem).strip()
    match = STEM_RE.fullmatch(text)
    if match is None:
        raise ValueError(f'Invalid canonical patch stem: {stem!r}')
    return match.group(1).lower(), int(match.group(2)), int(match.group(3))


def load_verified_split(label_name: str, expected_count: int):
    labels = np.load(VERIFIED / f'{label_name}_labels.npy', mmap_mode='r')
    manifest = load_json(VERIFIED / f'{label_name}_manifest.json')
    expected_shape = (expected_count, PATCH, PATCH)
    if tuple(labels.shape) != expected_shape:
        raise ValueError(f'{label_name}: labels shape {labels.shape} != {expected_shape}')
    if labels.dtype != np.uint8:
        raise TypeError(f'{label_name}: labels dtype {labels.dtype} != uint8')
    if len(manifest) != expected_count:
        raise ValueError(f'{label_name}: manifest count {len(manifest)} != {expected_count}')
    allowed = set(range(7)) | {IGNORE}
    values = set(np.unique(labels).tolist())
    if not values.issubset(allowed):
        raise ValueError(f'{label_name}: invalid label IDs {sorted(values)}')
    normalized=[]; seen=set()
    for i, raw in enumerate(manifest):
        if not isinstance(raw, dict):
            raise TypeError(f'{label_name}: manifest entry {i} is not an object')
        stem=str(raw.get('stem','')).strip()
        zone,row,col=parse_stem(stem)
        if stem in seen:
            raise RuntimeError(f'{label_name}: duplicate stem {stem!r}')
        seen.add(stem)
        explicit_zone=str(raw.get('zone','') or '').strip().lower()
        if explicit_zone and explicit_zone != zone:
            raise ValueError(f'{label_name}: explicit zone {explicit_zone!r} disagrees with stem {stem!r}')
        rec=dict(raw)
        rec['release_zone']=zone
        rec['release_row']=row
        rec['release_col']=col
        rec['release_stem']=stem
        normalized.append(rec)
    return normalized, np.asarray(labels), [r['release_stem'] for r in normalized]

val_manifest,val_labels,val_stems=load_verified_split('mh_val',30)
adapt_manifest,adapt_labels,adapt_stems=load_verified_split('mh_adapt',30)
test_manifest,test_labels,test_stems=load_verified_split('mh_test_partial',110)

all_stems=val_stems+adapt_stems+test_stems
if len(all_stems)!=len(set(all_stems)):
    raise RuntimeError('P4 splits are not patch-disjoint')

def count_zones(manifest):
    return dict(Counter(r['release_zone'] for r in manifest))

observed={'mh_val':count_zones(val_manifest),'mh_adapt':count_zones(adapt_manifest),'mh_test':count_zones(test_manifest)}
for name,spec in EXPECTED.items():
    if observed[name]!=spec['zones']:
        raise ValueError(f'{name}: zone counts {observed[name]} != expected {spec["zones"]}')

for split_name,manifest in [('mh_val',val_manifest),('mh_adapt',adapt_manifest)]:
    for rec in manifest:
        provenance=str(rec.get('split',rec.get('source_split','train'))).strip().lower()
        if provenance not in {'train','mh_val','mh_adapt'}:
            raise ValueError(f'{split_name}: invalid source provenance {provenance!r} for {rec["release_stem"]}')

print('✓ label shapes and domains PASS')
print('✓ manifest counts PASS')
print('✓ split stems are disjoint PASS')
print('✓ zone allocation PASS:', observed)
print('✓ canonical stem parsing PASS')
print('✓ release scope = 30 / 30 / 110 PASS')


✓ label shapes and domains PASS
✓ manifest counts PASS
✓ split stems are disjoint PASS
✓ zone allocation PASS: {'mh_val': {'pune': 15, 'satara': 15}, 'mh_adapt': {'pune': 15, 'satara': 15}, 'mh_test': {'pune': 40, 'satara': 40, 'vidarbha': 30}}
✓ canonical stem parsing PASS
✓ release scope = 30 / 30 / 110 PASS


In [3]:
# ============================================================
# CELL 3 — inspect the authoritative archive and resolve source arrays/meta
# ============================================================

with tarfile.open(ARCHIVE, 'r') as tar:
    members = [m for m in tar.getmembers() if m.isfile()]

basenames = defaultdict(list)
for m in members:
    basenames[Path(m.name).name].append(m)

required_arrays = [
    'pune_train.npy', 'pune_test.npy',
    'satara_train.npy', 'satara_test.npy',
    'vidarbha_test.npy',
]
required_names = required_arrays + ['norm_stats_trainonly.json']

missing = [name for name in required_names if not basenames.get(name)]
if missing:
    raise FileNotFoundError(
        'The authoritative processed archive is missing required source files:\n'
        + '\n'.join(f'  - {x}' for x in missing)
    )


def unique_member(basename):
    hits = basenames[basename]
    if len(hits) != 1:
        raise RuntimeError(
            f"Archive member '{basename}' is ambiguous. Found:\n" +
            '\n'.join('  - ' + m.name for m in hits)
        )
    return hits[0]

SOURCE_MEMBERS = {name: unique_member(name) for name in required_arrays + ['norm_stats_trainonly.json']}

# Metadata can exist in more than one historical representation. Collect every plausible
# zone/meta JSON and let the next cell decide which record list exactly matches each array.
META_MEMBER_CANDIDATES = []
for m in members:
    name = Path(m.name).name.lower()
    if not name.endswith('.json'):
        continue
    if any(name.startswith(zone + '_') for zone in ('pune', 'satara', 'vidarbha')) and 'meta' in name:
        META_MEMBER_CANDIDATES.append(m)

if not META_MEMBER_CANDIDATES:
    raise FileNotFoundError('No Pune/Satara/Vidarbha metadata JSON members were found in the processed archive.')

print('Required source arrays resolved:')
for name, member in SOURCE_MEMBERS.items():
    print(f'  {name:24s} <- {member.name}')
print('Metadata JSON candidates:', len(META_MEMBER_CANDIDATES))


Required source arrays resolved:
  pune_train.npy           <- proc/pune_train.npy
  pune_test.npy            <- proc/pune_test.npy
  satara_train.npy         <- proc/satara_train.npy
  satara_test.npy          <- proc/satara_test.npy
  vidarbha_test.npy        <- proc/vidarbha_test.npy
  norm_stats_trainonly.json <- proc/norm_stats_trainonly.json
Metadata JSON candidates: 6


In [4]:
# ============================================================
# CELL 4 — extract only required arrays, metadata, and norm stats
# ============================================================

EXTRACT = LOCAL_ROOT / 'source_extract'
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True, exist_ok=True)


def safe_extract_member(tar, member, root):
    # Prevent path traversal while extracting an authoritative archive.
    dest = (root / member.name).resolve()
    root_resolved = root.resolve()
    dest.relative_to(root_resolved)
    if member.isdir():
        dest.mkdir(parents=True, exist_ok=True)
        return
    if not member.isfile():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    source = tar.extractfile(member)
    if source is None:
        raise IOError(f'Unable to read archive member: {member.name}')
    with source, open(dest, 'wb') as out_f:
        shutil.copyfileobj(source, out_f)


wanted_members = list(SOURCE_MEMBERS.values()) + META_MEMBER_CANDIDATES
seen_names = set()

with tarfile.open(ARCHIVE, 'r') as tar:
    for member in wanted_members:
        if member.name in seen_names:
            continue
        seen_names.add(member.name)
        safe_extract_member(tar, member, EXTRACT)

# Resolve extracted paths by basename, retaining the original archive path when needed.
EXTRACTED_BY_BASENAME = defaultdict(list)
for p in EXTRACT.rglob('*'):
    if p.is_file():
        EXTRACTED_BY_BASENAME[p.name].append(p)

SOURCE_PATHS = {}
for basename in required_arrays + ['norm_stats_trainonly.json']:
    hits = EXTRACTED_BY_BASENAME.get(basename, [])
    if len(hits) != 1:
        raise RuntimeError(f'Extracted source resolution for {basename} is ambiguous/missing: {hits}')
    SOURCE_PATHS[basename] = hits[0]

META_PATHS = [
    p for p in EXTRACT.rglob('*.json')
    if 'meta' in p.name.lower()
    and any(p.name.lower().startswith(z + '_') for z in ('pune', 'satara', 'vidarbha'))
]

if not META_PATHS:
    raise RuntimeError('No extracted zone metadata files were found.')

# Copy train-only normalization stats as an independent P4 input artifact.
shutil.copy2(SOURCE_PATHS['norm_stats_trainonly.json'], LOCAL_ROOT / 'norm_stats_trainonly.json')
print('Extracted source arrays:', len(SOURCE_PATHS))
print('Extracted metadata files:', len(META_PATHS))


Extracted source arrays: 6
Extracted metadata files: 6


In [5]:
# ============================================================
# CELL 5 — authoritative source-array indexing and P4 stem resolution
# ============================================================
# IMPORTANT: the published source arrays are keyed by (zone, split), while
# the verified release labels use stems <zone>_<row>_<col>.  The source
# metadata may or may not contain a `stem` field, so this cell derives the
# canonical stem from zone/row/col and never requires the source metadata to
# carry one.

SOURCE_KEYS = (
    ('pune', 'train'), ('pune', 'test'),
    ('satara', 'train'), ('satara', 'test'),
    ('vidarbha', 'test'),
)


def zone_from_meta_filename(path: Path):
    name = path.name.lower()
    for zone in ZONE_NAMES:
        if name.startswith(zone + '_'):
            return zone
    raise ValueError(f'Cannot derive zone from metadata filename: {path.name}')


def filename_split(path: Path):
    name = path.name.lower()
    if name.endswith('_train_meta.json'):
        return 'train'
    if name.endswith('_test_meta.json'):
        return 'test'
    return None


def canonical_stem(zone, row, col):
    return f'{zone}_{int(row)}_{int(col)}'


def parse_record_coordinates(raw, path):
    if not isinstance(raw, dict):
        raise TypeError(f'{path.name}: metadata record is not an object')
    if 'row' not in raw or 'col' not in raw:
        raise KeyError(f'{path.name}: metadata record lacks row/col fields')
    row, col = int(raw['row']), int(raw['col'])
    if row < 0 or col < 0:
        raise ValueError(f'{path.name}: row/col must be non-negative, got {row}, {col}')
    return row, col


def normalize_meta_payload(payload, path):
    """Return records grouped as (zone, split) with split-local indices.

    Supported historical source forms:
      A) zone_meta.json = [records], records contain `split`;
      B) zone_meta.json = {'train': [...], 'test': [...]};
      C) zone_meta.json = {'records': [...]};
      D) zone_train_meta.json / zone_test_meta.json = [records];
      E) fallback: bare combined list with no `split`, partitioned by the
         verified source-array lengths for that zone.
    """
    zone = zone_from_meta_filename(path)
    fs = filename_split(path)

    groups = []
    if isinstance(payload, dict):
        # Standard combined dictionary.
        found = False
        for split in ('train', 'test'):
            value = payload.get(split)
            if isinstance(value, list):
                groups.append((split, value))
                found = True
        if not found:
            for key in ('records', 'meta'):
                value = payload.get(key)
                if isinstance(value, list):
                    if fs is None:
                        raise ValueError(f'{path.name}: {key} list has no split provenance')
                    groups.append((fs, value))
                    found = True
                    break
        if not found:
            # Some historical JSONs were dicts keyed by patch/stem. Flatten
            # dictionary values and use filename split or per-record split.
            values = [v for v in payload.values() if isinstance(v, dict)]
            if values:
                if fs is not None:
                    groups.append((fs, values))
                else:
                    by_split=defaultdict(list)
                    for rec in values:
                        split=str(rec.get('split','')).strip().lower()
                        if split in SOURCE_SPLITS:
                            by_split[split].append(rec)
                    groups.extend(sorted(by_split.items()))
        if not groups:
            raise ValueError(f'{path.name}: unsupported metadata dictionary structure')

    elif isinstance(payload, list):
        if not payload:
            return []
        has_split = all(
            isinstance(r, dict) and str(r.get('split','')).strip().lower() in SOURCE_SPLITS
            for r in payload
        )
        if has_split:
            by_split=defaultdict(list)
            for rec in payload:
                by_split[str(rec['split']).strip().lower()].append(rec)
            groups.extend(sorted(by_split.items()))
        elif fs is not None:
            groups.append((fs, payload))
        else:
            # Final historical fallback for a combined zone_meta.json with no
            # split field: source array lengths define the boundary exactly.
            train_n = ARRAY_INFO[(zone,'train')]['n'] if (zone,'train') in ARRAY_INFO else 0
            test_n = ARRAY_INFO[(zone,'test')]['n'] if (zone,'test') in ARRAY_INFO else 0
            if len(payload) != train_n + test_n:
                raise ValueError(
                    f'{path.name}: bare metadata list has {len(payload)} records, '
                    f'but source arrays require train={train_n}, test={test_n}'
                )
            groups.append(('train', payload[:train_n]))
            if test_n:
                groups.append(('test', payload[train_n:]))
    else:
        raise TypeError(f'{path.name}: unsupported JSON payload type {type(payload).__name__}')

    normalized=[]
    for split, records in groups:
        if split not in SOURCE_SPLITS:
            continue
        for local_index, raw in enumerate(records):
            row, col = parse_record_coordinates(raw, path)
            rec_zone=str(raw.get('zone', zone)).strip().lower()
            if rec_zone != zone:
                raise ValueError(f'{path.name}: record zone {rec_zone!r} != filename zone {zone!r}')
            rec_split=str(raw.get('split', split)).strip().lower()
            if rec_split != split:
                raise ValueError(f'{path.name}: record split {rec_split!r} != group split {split!r}')
            stem=canonical_stem(zone,row,col)
            raw_stem=str(raw.get('stem','') or '').strip()
            if raw_stem and raw_stem != stem:
                raise ValueError(f'{path.name}: source stem {raw_stem!r} != canonical {stem!r}')
            normalized.append({
                'zone': zone,
                'split': split,
                'row': row,
                'col': col,
                'stem': stem,
                'index': local_index,
                'meta_file': path.name,
            })
    return normalized


# Validate every source tensor first.  This also guarantees PATCH_SHAPE is
# defined before it is referenced.
ARRAY_INFO={}
for zone, split in SOURCE_KEYS:
    basename=f'{zone}_{split}.npy'
    if basename not in SOURCE_PATHS:
        raise FileNotFoundError(f'Source array missing after extraction: {basename}')
    arr=np.load(SOURCE_PATHS[basename], mmap_mode='r')
    if arr.dtype != np.int16:
        raise TypeError(f'{basename}: expected int16, got {arr.dtype}')
    if arr.ndim != 5 or tuple(arr.shape[1:]) != PATCH_SHAPE:
        raise ValueError(f'{basename}: expected shape [N,2,17,128,128], got {arr.shape}')
    ARRAY_INFO[(zone,split)]={'path':SOURCE_PATHS[basename], 'n':int(arr.shape[0]), 'shape':tuple(arr.shape)}

# Parse every metadata candidate and deduplicate identical historical copies.
RECORDS_BY_COORD={}
RECORDS_BY_STEM={}
metadata_stats=[]
for meta_path in META_PATHS:
    payload=load_json(meta_path)
    records=normalize_meta_payload(payload, meta_path)
    metadata_stats.append((meta_path.name,len(records)))
    grouped=defaultdict(list)
    for rec in records:
        grouped[(rec['zone'],rec['split'])].append(rec)

    for key, group in grouped.items():
        if key not in ARRAY_INFO:
            continue
        expected_n=ARRAY_INFO[key]['n']
        if len(group) != expected_n:
            raise ValueError(
                f'{meta_path.name}: metadata count {len(group)} for {key} != source array length {expected_n}'
            )
        for rec in group:
            coord=(rec['zone'],rec['split'],rec['row'],rec['col'])
            old=RECORDS_BY_COORD.get(coord)
            if old is not None:
                if (old['index'],old['stem']) != (rec['index'],rec['stem']):
                    raise RuntimeError(f'Conflicting source coordinate mapping: {coord}: {old} vs {rec}')
            else:
                RECORDS_BY_COORD[coord]=rec
            old_stem=RECORDS_BY_STEM.get(rec['stem'])
            if old_stem is not None:
                if (old_stem['zone'],old_stem['split'],old_stem['row'],old_stem['col'],old_stem['index']) != (
                    rec['zone'],rec['split'],rec['row'],rec['col'],rec['index']):
                    raise RuntimeError(f'Conflicting source stem mapping: {rec["stem"]}: {old_stem} vs {rec}')
            else:
                RECORDS_BY_STEM[rec['stem']]=rec

print('Metadata parse summary:', metadata_stats)
print('Indexed unique source coordinates:', len(RECORDS_BY_COORD))
print('Indexed unique source stems     :', len(RECORDS_BY_STEM))
if not RECORDS_BY_COORD:
    raise RuntimeError('No source patch metadata records were indexed from the authoritative archive')

# Every source patch must be covered exactly once after deduplication.
for key, info in ARRAY_INFO.items():
    covered=sum(1 for rec in RECORDS_BY_COORD.values() if (rec['zone'],rec['split'])==key)
    if covered != info['n']:
        raise RuntimeError(f'{key}: metadata coverage {covered} != source patches {info["n"]}')

# Resolve verified label manifests using coordinates parsed from their stems.
for split_name, manifest, source_split in [
    ('mh_val',val_manifest,'train'),
    ('mh_adapt',adapt_manifest,'train'),
    ('mh_test',test_manifest,'test'),
]:
    for rec in manifest:
        key=(rec['release_zone'],source_split,rec['release_row'],rec['release_col'])
        source=RECORDS_BY_COORD.get(key)
        if source is None:
            raise KeyError(f'{split_name}: verified stem {rec["release_stem"]} not found in source metadata for key {key}')
        if source['stem'] != rec['release_stem']:
            raise RuntimeError(f'{split_name}: source stem {source["stem"]} != verified stem {rec["release_stem"]}')
        rec['source_index']=int(source['index'])
        rec['source_split']=source['split']
        rec['source_meta_file']=source['meta_file']

print('✓ authoritative source metadata indexed PASS')
print('✓ verified 30/30/110 stems resolved by zone/split/row/col PASS')
print('✓ no source stem field required PASS')


Metadata parse summary: [('pune_train_meta.json', 3315), ('vidarbha_train_meta.json', 0), ('vidarbha_test_meta.json', 1120), ('pune_test_meta.json', 817), ('satara_test_meta.json', 820), ('satara_train_meta.json', 3349)]
Indexed unique source coordinates: 9421
Indexed unique source stems     : 9421
✓ authoritative source metadata indexed PASS
✓ verified 30/30/110 stems resolved by zone/split/row/col PASS
✓ no source stem field required PASS


In [6]:
# ============================================================
# CELL 6 — materialize exact P4 arrays in verified-manifest order
# ============================================================
def materialize_split(split_name, manifest, labels):
    output=np.empty((len(manifest),)+PATCH_SHAPE,dtype=np.int16)
    output_manifest=[]
    opened={}
    for i, rec in enumerate(manifest):
        key=(rec['release_zone'],rec['source_split'])
        source_path=ARRAY_INFO[key]['path']
        if source_path not in opened:
            opened[source_path]=np.load(source_path,mmap_mode='r')
        raw=np.asarray(opened[source_path][int(rec['source_index'])],dtype=np.int16)
        if tuple(raw.shape)!=PATCH_SHAPE:
            raise ValueError(f'{split_name}/{rec["release_stem"]}: raw patch shape {raw.shape} != {PATCH_SHAPE}')
        output[i]=raw
        output_manifest.append(dict(rec))
    y=np.asarray(labels,dtype=np.uint8)
    expected_y=(len(manifest),PATCH,PATCH)
    if tuple(y.shape)!=expected_y:
        raise ValueError(f'{split_name}: label shape {y.shape} != {expected_y}')
    return output,y,output_manifest

val_x,val_y,val_out_manifest=materialize_split('mh_val',val_manifest,val_labels)
adapt_x,adapt_y,adapt_out_manifest=materialize_split('mh_adapt',adapt_manifest,adapt_labels)
test_x,test_y,test_out_manifest=materialize_split('mh_test',test_manifest,test_labels)

for name,x,y in [('mh_val',val_x,val_y),('mh_adapt',adapt_x,adapt_y),('mh_test',test_x,test_y)]:
    if x.dtype!=np.int16 or y.dtype!=np.uint8:
        raise TypeError(f'{name}: expected int16 tensors + uint8 labels, got {x.dtype}, {y.dtype}')
    if x.ndim!=5 or tuple(x.shape[1:])!=PATCH_SHAPE:
        raise ValueError(f'{name}: unexpected tensor shape {x.shape}')
    if not np.isfinite(x.astype(np.float32)).all():
        raise FloatingPointError(f'{name}: non-finite tensor values')
    if not set(np.unique(y).tolist()).issubset(set(range(7))|{IGNORE}):
        raise ValueError(f'{name}: invalid label IDs')

np.save(LOCAL_ROOT/'p4_mh_val.npy',val_x)
np.save(LOCAL_ROOT/'p4_mh_val_labels.npy',val_y)
(LOCAL_ROOT/'p4_mh_val_manifest.json').write_text(json.dumps(val_out_manifest,indent=2),encoding='utf-8')
np.save(LOCAL_ROOT/'p4_mh_adapt.npy',adapt_x)
np.save(LOCAL_ROOT/'p4_mh_adapt_labels.npy',adapt_y)
(LOCAL_ROOT/'p4_mh_adapt_manifest.json').write_text(json.dumps(adapt_out_manifest,indent=2),encoding='utf-8')
np.save(LOCAL_ROOT/'p4_mh_test.npy',test_x)
np.save(LOCAL_ROOT/'p4_mh_test_labels.npy',test_y)
(LOCAL_ROOT/'p4_mh_test_manifest.json').write_text(json.dumps(test_out_manifest,indent=2),encoding='utf-8')

print('✓ exact verified-manifest order materialized')
print('  MH-VAL  :',val_x.shape)
print('  MH-ADAPT:',adapt_x.shape)
print('  MH-TEST :',test_x.shape)


✓ exact verified-manifest order materialized
  MH-VAL  : (30, 2, 17, 128, 128)
  MH-ADAPT: (30, 2, 17, 128, 128)
  MH-TEST : (110, 2, 17, 128, 128)


In [7]:
# ============================================================
# CELL 7 — independent content/lineage checks and release metadata
# ============================================================

# q is stored in raw int16 with ARRAY_SCALE=10000 and should therefore be in [0,1].
with open(LOCAL_ROOT / 'norm_stats_trainonly.json', 'r', encoding='utf-8') as f:
    norm_stats = json.load(f)

if list(norm_stats.get('sar_channels', [])) != [13, 14, 15]:
    raise ValueError(f"Unexpected SAR channel contract: {norm_stats.get('sar_channels')}")
if float(norm_stats.get('array_scale', -1)) != 10000.0:
    raise ValueError(f"Unexpected array scale: {norm_stats.get('array_scale')}")
if float(norm_stats.get('sar_extra_scale', -1)) != 100.0:
    raise ValueError(f"Unexpected SAR extra scale: {norm_stats.get('sar_extra_scale')}")

for name, x in [('VAL', val_x), ('ADAPT', adapt_x), ('TEST', test_x)]:
    q = x[:, :, 16].astype(np.float32) / 10000.0
    if q.min() < -1e-5 or q.max() > 1.00001:
        raise ValueError(f'{name}: q outside [0,1]: min={q.min()} max={q.max()}')

release_scope = {
    'project': 'Geo-Nexus v3.2',
    'stage': 'P4_MH_ZS_FS',
    'dataset_id': KAGGLE_DATASET_ID,
    'tensor_dtype': 'int16',
    'tensor_shape_per_patch': [2, 17, 128, 128],
    'splits': {
        'mh_val': {'count': 30, 'zones': {'pune': 15, 'satara': 15}, 'source': 'verified human-reviewed label manifest + train-AOI imagery'},
        'mh_adapt': {'count': 30, 'zones': {'pune': 15, 'satara': 15}, 'source': 'verified human-reviewed label manifest + train-AOI imagery'},
        'mh_test': {'count': 110, 'zones': {'pune': 40, 'satara': 40, 'vidarbha': 30}, 'source': 'verified human-reviewed label manifest + held-out imagery'},
    },
    'excluded_from_current_verified_test': ['MH-TEST-BLIND (20)', 'MH-TEST-MONSOON (30)'],
    'label_status': 'human-reviewed weak labels; not independently redrawn ground truth',
    'normalization': 'norm_stats_trainonly.json from TRAIN AOI only',
    'source_archive': ARCHIVE.name,
}
(LOCAL_ROOT / 'release_scope.json').write_text(json.dumps(release_scope, indent=2), encoding='utf-8')

readme = """# Geo-Nexus v3.2 — P4 Maharashtra Evaluation Source

This dataset is a compact, P4-specific evaluation source materialized from the authoritative processed Maharashtra archive and the current verified-label release.

## Files
- `p4_mh_val.npy` / `p4_mh_val_labels.npy` / `p4_mh_val_manifest.json` — 30
- `p4_mh_adapt.npy` / `p4_mh_adapt_labels.npy` / `p4_mh_adapt_manifest.json` — 30
- `p4_mh_test.npy` / `p4_mh_test_labels.npy` / `p4_mh_test_manifest.json` — 110
- `norm_stats_trainonly.json`
- `release_scope.json`
- `SHA256SUMS.json`

## Important scope
The current verified release contains 110 test patches: 40 Pune dry + 40 Satara dry + 30 Vidarbha. The 20 blind and 30 monsoon masks are intentionally excluded and are not fabricated here.

The 30 MH-VAL and 30 MH-ADAPT masks are human-reviewed automatic labels. They are not independently redrawn clean ground truth.

The tensors are stored as int16 using the project's array-scale/SAR-scale convention; P4 decodes them with `array_scale=10000` and `sar_extra_scale=100` for channels 13-15.
"""
(LOCAL_ROOT / 'README.md').write_text(readme, encoding='utf-8')

# Self-test the exact counts and zone composition again after writing.
for name, expected_count in [('p4_mh_val', 30), ('p4_mh_adapt', 30), ('p4_mh_test', 110)]:
    arr = np.load(LOCAL_ROOT / f'{name}.npy', mmap_mode='r')
    lab = np.load(LOCAL_ROOT / f'{name}_labels.npy', mmap_mode='r')
    assert arr.shape[0] == expected_count
    assert arr.shape[1:] == EXPECTED_ARRAY_SHAPE
    assert lab.shape == (expected_count, PATCH, PATCH)

print('✓ normalization contract PASS')
print('✓ q range PASS')
print('✓ post-write shapes/counts PASS')
print('✓ current 110-patch verified test scope preserved PASS')


✓ normalization contract PASS
✓ q range PASS
✓ post-write shapes/counts PASS
✓ current 110-patch verified test scope preserved PASS


In [8]:
# ============================================================
# CELL 8 — SHA256 manifest
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

files = sorted(p for p in LOCAL_ROOT.iterdir() if p.is_file() and p.name != 'SHA256SUMS.json')
sha = {p.name: {'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in files}
(LOCAL_ROOT / 'SHA256SUMS.json').write_text(json.dumps(sha, indent=2), encoding='utf-8')

print('P4 evaluation-source files:')
for name, meta in sha.items():
    print(f"  {name:34s} {meta['bytes']:12d} bytes  {meta['sha256'][:16]}...")


P4 evaluation-source files:
  README.md                                  1057 bytes  f5242d9058bccaa1...
  norm_stats_trainonly.json                   973 bytes  eea56b759f6341c9...
  p4_mh_adapt.npy                        33423488 bytes  d34c274358e8157e...
  p4_mh_adapt_labels.npy                   491648 bytes  1c11cb70238583f9...
  p4_mh_adapt_manifest.json                 16801 bytes  84d6e135e08a7fa7...
  p4_mh_test.npy                        122552448 bytes  307461b2d12de00c...
  p4_mh_test_labels.npy                   1802368 bytes  6eb1749fba3ce26b...
  p4_mh_test_manifest.json                  62530 bytes  218cb7b05ec9141c...
  p4_mh_val.npy                          33423488 bytes  17431c921d392861...
  p4_mh_val_labels.npy                     491648 bytes  192280ae440eb03f...
  p4_mh_val_manifest.json                   16773 bytes  571b0cc65d2af4be...
  release_scope.json                         1117 bytes  8295dafa5e54b438...


## Pre-publication audit status

This notebook was statically audited after the manifest-contract correction. All code cells must pass Python AST parsing before publication. The data-dependent archive/label assertions require the user's Colab Drive source artifacts and therefore are intentionally executed in Colab rather than mocked here.


In [9]:
# ============================================================
# CELL 9 — final on-disk release self-test
# ============================================================
EXPECTED_FILES = {
    'p4_mh_val.npy', 'p4_mh_val_labels.npy', 'p4_mh_val_manifest.json',
    'p4_mh_adapt.npy', 'p4_mh_adapt_labels.npy', 'p4_mh_adapt_manifest.json',
    'p4_mh_test.npy', 'p4_mh_test_labels.npy', 'p4_mh_test_manifest.json',
    'norm_stats_trainonly.json', 'release_scope.json', 'README.md', 'SHA256SUMS.json',
}
actual_files = {p.name for p in LOCAL_ROOT.iterdir() if p.is_file()}
missing_final = sorted(EXPECTED_FILES - actual_files)
if missing_final:
    raise RuntimeError(f'Missing final release files: {missing_final}')

for name, expected_count in [('p4_mh_val', 30), ('p4_mh_adapt', 30), ('p4_mh_test', 110)]:
    x = np.load(LOCAL_ROOT / f'{name}.npy', mmap_mode='r')
    y = np.load(LOCAL_ROOT / f'{name}_labels.npy', mmap_mode='r')
    manifest = load_json(LOCAL_ROOT / f'{name}_manifest.json')
    if tuple(x.shape) != (expected_count,) + PATCH_SHAPE:
        raise ValueError(f'{name}: final tensor shape {x.shape} is incorrect')
    if tuple(y.shape) != (expected_count, PATCH, PATCH):
        raise ValueError(f'{name}: final label shape {y.shape} is incorrect')
    if len(manifest) != expected_count:
        raise ValueError(f'{name}: final manifest count {len(manifest)} != {expected_count}')
    stems = [m['stem'] for m in manifest]
    if len(stems) != len(set(stems)):
        raise RuntimeError(f'{name}: final manifest contains duplicate stems')

all_final_manifests = [load_json(LOCAL_ROOT / f'{name}_manifest.json') for name in ('p4_mh_val', 'p4_mh_adapt', 'p4_mh_test')]
all_final_stems = [m['stem'] for manifest in all_final_manifests for m in manifest]
if len(all_final_stems) != len(set(all_final_stems)):
    raise RuntimeError('Final P4 output manifests are not disjoint')

print('=' * 80)
print('FINAL P4 EVALUATION-SOURCE SELF-TEST: PASS')
print('=' * 80)
print('30 MH-VAL + 30 MH-ADAPT + 110 MH-TEST')
print('All required files present.')
print('All final tensor/label/manifest shapes verified.')
print('All final stems are unique across all P4 splits.')


FINAL P4 EVALUATION-SOURCE SELF-TEST: PASS
30 MH-VAL + 30 MH-ADAPT + 110 MH-TEST
All required files present.
All final tensor/label/manifest shapes verified.
All final stems are unique across all P4 splits.


## Optional Kaggle publication

Set `UPLOAD_TO_KAGGLE = True` in Cell 1 only after the local staging/self-test passes.

The upload target is:

```text
sumit07125/geonexus-mh-p4-eval-v3-2
```

This is a **new compact evaluation dataset**. It does not replace `sumit07125/geonexus-mh-v3` because that dataset remains the DAPT train dataset.

In [13]:
# ============================================================
# CELL 10 — FINAL KAGGLE PUBLICATION
# Current Kaggle token + legacy kaggle.json support
# Colab-safe authentication and CREATE/VERSION detection
# ============================================================

import os
import sys
import json
import glob
import shutil
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# Publication switch
# ------------------------------------------------------------
if not UPLOAD_TO_KAGGLE:
    print("UPLOAD_TO_KAGGLE=False — local P4 evaluation source is ready.")

else:

    print("=" * 80)
    print("KAGGLE PUBLICATION — Geo-Nexus P4 Evaluation Source")
    print("=" * 80)

    # ========================================================
    # 1. Verify the already-built P4 release
    # ========================================================
    required_files = {
        "p4_mh_val.npy",
        "p4_mh_val_labels.npy",
        "p4_mh_val_manifest.json",
        "p4_mh_adapt.npy",
        "p4_mh_adapt_labels.npy",
        "p4_mh_adapt_manifest.json",
        "p4_mh_test.npy",
        "p4_mh_test_labels.npy",
        "p4_mh_test_manifest.json",
        "norm_stats_trainonly.json",
        "release_scope.json",
        "README.md",
        "SHA256SUMS.json",
    }

    missing = sorted(
        name for name in required_files
        if not (LOCAL_ROOT / name).is_file()
    )

    if missing:
        raise FileNotFoundError(
            "P4 release is incomplete. Missing files:\n"
            + "\n".join(f"  - {x}" for x in missing)
        )

    print("✓ P4 release files verified")

    # ========================================================
    # 2. Install / verify Kaggle CLI
    # ========================================================
    print("Checking Kaggle CLI...")

    cli_check = subprocess.run(
        [sys.executable, "-m", "kaggle", "--version"],
        text=True,
        capture_output=True,
    )

    if cli_check.returncode != 0:
        print("Kaggle CLI not available. Installing/updating...")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-U",
                "kaggle",
            ],
            check=True,
        )

    # ========================================================
    # 3. Prepare authentication environment
    #
    # Priority:
    #   A. existing KAGGLE_API_TOKEN
    #   B. existing access_token file
    #   C. legacy kaggle.json
    #   D. scan uploaded Colab files
    #   E. interactively upload credential file
    # ========================================================
    kaggle_env = os.environ.copy()
    auth_source = None

    # --------------------------------------------------------
    # A. Already-defined environment token
    # --------------------------------------------------------
    existing_token = os.environ.get("KAGGLE_API_TOKEN", "").strip()

    if existing_token:
        kaggle_env["KAGGLE_API_TOKEN"] = existing_token
        auth_source = "KAGGLE_API_TOKEN environment variable"

    # --------------------------------------------------------
    # B. Search token files
    # --------------------------------------------------------
    if auth_source is None:

        token_candidates = [
            Path.home() / ".kaggle" / "access_token",
            Path.home() / ".kaggle" / "access_token.txt",
            Path("/content/access_token"),
            Path("/content/access_token.txt"),
            Path("/content/kaggle_token"),
            Path("/content/kaggle_token.txt"),
        ]

        token_path = next(
            (p for p in token_candidates if p.is_file()),
            None
        )

        if token_path is not None:
            token_value = token_path.read_text(
                encoding="utf-8",
                errors="ignore"
            ).strip()

            if token_value:
                kaggle_env["KAGGLE_API_TOKEN"] = token_value
                auth_source = f"token file: {token_path}"

    # --------------------------------------------------------
    # C. Search legacy kaggle.json
    # --------------------------------------------------------
    if auth_source is None:

        legacy_candidates = [
            Path.home() / ".kaggle" / "kaggle.json",
            Path("/content/kaggle.json"),
        ]

        legacy_path = next(
            (p for p in legacy_candidates if p.is_file()),
            None
        )

        if legacy_path is not None:

            try:
                with open(legacy_path, "r", encoding="utf-8") as f:
                    legacy_data = json.load(f)
            except Exception as exc:
                raise RuntimeError(
                    f"Could not read {legacy_path}: {exc}"
                )

            if (
                isinstance(legacy_data, dict)
                and "username" in legacy_data
                and "key" in legacy_data
            ):
                kaggle_dir = Path.home() / ".kaggle"
                kaggle_dir.mkdir(parents=True, exist_ok=True)

                target = kaggle_dir / "kaggle.json"

                if legacy_path.resolve() != target.resolve():
                    shutil.copy2(legacy_path, target)

                try:
                    os.chmod(target, 0o600)
                except Exception:
                    pass

                kaggle_env["KAGGLE_USERNAME"] = str(
                    legacy_data["username"]
                )
                kaggle_env["KAGGLE_KEY"] = str(
                    legacy_data["key"]
                )

                auth_source = f"legacy kaggle.json: {legacy_path}"

    # ========================================================
    # 4. Search all files currently uploaded to /content
    #
    # This handles the common Colab case where the uploaded
    # filename is not exactly "kaggle.json".
    # ========================================================
    if auth_source is None:

        discovered_files = []

        for pattern in [
            "/content/*",
            "/content/**/*",
        ]:
            discovered_files.extend(
                Path(p)
                for p in glob.glob(pattern, recursive=True)
                if Path(p).is_file()
            )

        # Remove duplicates while preserving order.
        unique_files = []
        seen_paths = set()

        for p in discovered_files:
            try:
                rp = p.resolve()
            except Exception:
                rp = p

            if rp not in seen_paths:
                seen_paths.add(rp)
                unique_files.append(p)

        # ----------------------------------------------------
        # Look for legacy credential JSON
        # ----------------------------------------------------
        for candidate in unique_files:

            if candidate.name in {
                "dataset-metadata.json",
                "SHA256SUMS.json",
            }:
                continue

            if candidate.suffix.lower() != ".json":
                continue

            try:
                with open(candidate, "r", encoding="utf-8") as f:
                    data = json.load(f)
            except Exception:
                continue

            if (
                isinstance(data, dict)
                and "username" in data
                and "key" in data
            ):
                kaggle_dir = Path.home() / ".kaggle"
                kaggle_dir.mkdir(parents=True, exist_ok=True)

                target = kaggle_dir / "kaggle.json"
                shutil.copy2(candidate, target)

                try:
                    os.chmod(target, 0o600)
                except Exception:
                    pass

                kaggle_env["KAGGLE_USERNAME"] = str(
                    data["username"]
                )
                kaggle_env["KAGGLE_KEY"] = str(
                    data["key"]
                )

                auth_source = f"legacy credential file: {candidate}"
                break

        # ----------------------------------------------------
        # Look for a JSON file containing a new-style token
        # ----------------------------------------------------
        if auth_source is None:

            for candidate in unique_files:

                if candidate.suffix.lower() != ".json":
                    continue

                try:
                    with open(candidate, "r", encoding="utf-8") as f:
                        data = json.load(f)
                except Exception:
                    continue

                if not isinstance(data, dict):
                    continue

                possible_token = (
                    data.get("access_token")
                    or data.get("token")
                    or data.get("KAGGLE_API_TOKEN")
                )

                if isinstance(possible_token, str):
                    possible_token = possible_token.strip()

                    if possible_token:
                        kaggle_env["KAGGLE_API_TOKEN"] = possible_token
                        auth_source = f"token JSON file: {candidate}"
                        break

    # ========================================================
    # 5. If still not authenticated, ask user to upload token
    # ========================================================
    if auth_source is None:

        print()
        print("=" * 80)
        print("KAGGLE CREDENTIAL FILE NOT FOUND IN THE RUNTIME")
        print("=" * 80)
        print()
        print(
            "Select your Kaggle credential/token file in the upload dialog."
        )
        print(
            "The token itself will NOT be printed."
        )
        print()

        try:
            from google.colab import files as colab_files
        except ImportError:
            raise RuntimeError(
                "This notebook is expected to run in Google Colab, "
                "but google.colab.files is unavailable."
            )

        uploaded = colab_files.upload()

        if not uploaded:
            raise RuntimeError(
                "No Kaggle credential file was uploaded."
            )

        # ----------------------------------------------------
        # Parse the uploaded file(s)
        # ----------------------------------------------------
        for filename in uploaded.keys():

            candidate = Path(filename)

            # ----------------------------------------------
            # JSON credential
            # ----------------------------------------------
            if candidate.suffix.lower() == ".json":

                try:
                    with open(candidate, "r", encoding="utf-8") as f:
                        data = json.load(f)
                except Exception:
                    data = None

                if isinstance(data, dict):

                    # Legacy format
                    if (
                        "username" in data
                        and "key" in data
                    ):
                        kaggle_dir = Path.home() / ".kaggle"
                        kaggle_dir.mkdir(
                            parents=True,
                            exist_ok=True
                        )

                        target = kaggle_dir / "kaggle.json"
                        shutil.copy2(candidate, target)

                        try:
                            os.chmod(target, 0o600)
                        except Exception:
                            pass

                        kaggle_env["KAGGLE_USERNAME"] = str(
                            data["username"]
                        )
                        kaggle_env["KAGGLE_KEY"] = str(
                            data["key"]
                        )

                        auth_source = f"uploaded legacy file: {candidate}"
                        break

                    # New token embedded in JSON
                    possible_token = (
                        data.get("access_token")
                        or data.get("token")
                        or data.get("KAGGLE_API_TOKEN")
                    )

                    if isinstance(possible_token, str):
                        possible_token = possible_token.strip()

                        if possible_token:
                            kaggle_env["KAGGLE_API_TOKEN"] = possible_token
                            auth_source = f"uploaded token JSON: {candidate}"
                            break

            # ----------------------------------------------
            # Plain-text token
            # ----------------------------------------------
            try:
                raw_text = candidate.read_text(
                    encoding="utf-8",
                    errors="ignore"
                ).strip()
            except Exception:
                raw_text = ""

            if raw_text:

                # Current Kaggle access tokens begin with KGAT.
                if raw_text.startswith("KGAT"):
                    kaggle_env["KAGGLE_API_TOKEN"] = raw_text
                    auth_source = f"uploaded access token: {candidate}"
                    break

    # ========================================================
    # 6. Final authentication check
    # ========================================================
    if auth_source is None:
        raise RuntimeError(
            "Kaggle authentication could not be detected.\n\n"
            "Upload either:\n"
            "  • the current Kaggle access-token file/string, or\n"
            "  • a legacy kaggle.json containing username + key."
        )

    print("✓ Kaggle authentication detected")
    print("  Source:", auth_source)

    # --------------------------------------------------------
    # Verify credentials BEFORE attempting publication.
    # --------------------------------------------------------
    auth_test = subprocess.run(
        [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "list",
            "-s",
            "geonexus",
        ],
        text=True,
        capture_output=True,
        env=kaggle_env,
    )

    if auth_test.returncode != 0:

        print()
        print("Kaggle authentication test FAILED.")
        print()

        if auth_test.stdout:
            print(auth_test.stdout)

        if auth_test.stderr:
            print(auth_test.stderr)

        raise RuntimeError(
            "Kaggle credentials were found, but Kaggle rejected them. "
            "Generate/copy a fresh Kaggle API token and run Cell 10 again."
        )

    print("✓ Kaggle authentication test PASS")

    # ========================================================
    # 7. Write dataset metadata
    # ========================================================
    metadata = {
        "title": "Geo-Nexus v3.2 — P4 Maharashtra Evaluation Source",
        "id": KAGGLE_DATASET_ID,
        "licenses": [{"name": "other"}],
        "description": (
            "Compact P4 zero-shot/few-shot Maharashtra evaluation source "
            "derived from the authoritative Geo-Nexus v3.2 processed archive "
            "and verified 30/30/110 label manifests."
        ),
    }

    (LOCAL_ROOT / "dataset-metadata.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

    print("✓ dataset-metadata.json written")

    # ========================================================
    # 8. Check whether P4 dataset already exists
    # ========================================================
    status = subprocess.run(
        [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "status",
            KAGGLE_DATASET_ID,
        ],
        text=True,
        capture_output=True,
        env=kaggle_env,
    )

    dataset_exists = (status.returncode == 0)

    # ========================================================
    # 9. CREATE or VERSION
    # ========================================================
    if dataset_exists:

        print("✓ Existing P4 Kaggle dataset detected")
        print("  Action: create new version")

        upload_command = [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "version",
            "-p",
            str(LOCAL_ROOT),
            "-m",
            "Geo-Nexus v3.2 P4 Maharashtra evaluation source",
            "--dir-mode",
            "zip",
        ]

    else:

        print("✓ P4 Kaggle dataset does not yet exist")
        print("  Action: create new dataset")

        upload_command = [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "create",
            "-p",
            str(LOCAL_ROOT),
            "--dir-mode",
            "zip",
        ]

    # ========================================================
    # 10. Upload
    # ========================================================
    print()
    print("Uploading:", KAGGLE_DATASET_ID)
    print()

    result = subprocess.run(
        upload_command,
        text=True,
        capture_output=True,
        env=kaggle_env,
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:

        if result.stderr:
            print()
            print("Kaggle publication error:")
            print(result.stderr)

        raise RuntimeError(
            "Kaggle publication failed."
        )

    print()
    print("✓ Kaggle publication command completed")

    # ========================================================
    # 11. Final verification
    # ========================================================
    verify = subprocess.run(
        [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "status",
            KAGGLE_DATASET_ID,
        ],
        text=True,
        capture_output=True,
        env=kaggle_env,
    )

    if verify.returncode != 0:
        print(verify.stdout)
        print(verify.stderr)
        raise RuntimeError(
            "Upload completed, but final Kaggle verification failed."
        )

    print()
    print("=" * 80)
    print("P4 KAGGLE PUBLICATION: PASS")
    print("=" * 80)
    print("Dataset:", KAGGLE_DATASET_ID)
    print(
        "URL:",
        f"https://www.kaggle.com/datasets/{KAGGLE_DATASET_ID}"
    )
    print()
    print("Scope:")
    print("  MH-VAL   : 30")
    print("  MH-ADAPT : 30")
    print("  MH-TEST  : 110")
    print()
    print(verify.stdout)

KAGGLE PUBLICATION — Geo-Nexus P4 Evaluation Source
✓ P4 release files verified
Checking Kaggle CLI...

KAGGLE CREDENTIAL FILE NOT FOUND IN THE RUNTIME

Select your Kaggle credential/token file in the upload dialog.
The token itself will NOT be printed.



Saving kaggle.json to kaggle.json
✓ Kaggle authentication detected
  Source: uploaded legacy file: kaggle.json
✓ Kaggle authentication test PASS
✓ dataset-metadata.json written
✓ P4 Kaggle dataset does not yet exist
  Action: create new dataset

Uploading: sumit07125/geonexus-mh-p4-eval-v3-2

Starting upload for file p4_mh_test.npy
Upload successful: p4_mh_test.npy (117MB)
Starting upload for file p4_mh_val_manifest.json
Upload successful: p4_mh_val_manifest.json (16KB)
Starting upload for file p4_mh_test_manifest.json
Upload successful: p4_mh_test_manifest.json (61KB)
Starting upload for file SHA256SUMS.json
Upload successful: SHA256SUMS.json (2KB)
Starting upload for file p4_mh_adapt_labels.npy
Upload successful: p4_mh_adapt_labels.npy (480KB)
Starting upload for file norm_stats_trainonly.json
Upload successful: norm_stats_trainonly.json (973B)
Starting upload for file p4_mh_val_labels.npy
Upload successful: p4_mh_val_labels.npy (480KB)
Starting upload for file p4_mh_adapt_manifest.j

RuntimeError: Upload completed, but final Kaggle verification failed.

In [14]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "kaggle",
    "datasets",
    "version",
    "-p",
    str(LOCAL_ROOT),
    "-m",
    "Geo-Nexus v3.2 P4 clean evaluation source",
    "--dir-mode",
    "skip",
]

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
    env=kaggle_env,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Clean Kaggle version upload failed.")

print("✅ CLEAN P4 DATASET VERSION UPLOADED")
print("https://www.kaggle.com/datasets/sumit07125/geonexus-mh-p4-eval-v3-2")

Starting upload for file p4_mh_test.npy
Upload successful: p4_mh_test.npy (117MB)
Starting upload for file p4_mh_val_manifest.json
Upload successful: p4_mh_val_manifest.json (16KB)
Starting upload for file p4_mh_test_manifest.json
Upload successful: p4_mh_test_manifest.json (61KB)
Starting upload for file SHA256SUMS.json
Upload successful: SHA256SUMS.json (2KB)
Starting upload for file p4_mh_adapt_labels.npy
Upload successful: p4_mh_adapt_labels.npy (480KB)
Starting upload for file norm_stats_trainonly.json
Upload successful: norm_stats_trainonly.json (973B)
Starting upload for file p4_mh_val_labels.npy
Upload successful: p4_mh_val_labels.npy (480KB)
Starting upload for file p4_mh_adapt_manifest.json
Upload successful: p4_mh_adapt_manifest.json (16KB)
Skipping folder: source_extract; use '--dir-mode' to upload folders
Starting upload for file p4_mh_val.npy
Upload successful: p4_mh_val.npy (32MB)
Starting upload for file p4_mh_test_labels.npy
Upload successful: p4_mh_test_labels.npy (2M